In [9]:
!pip install pandas numpy scikit-learn fairlearn pgmpy

In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import os

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [11]:
def preprocess_adult_for_fair_bbn(csv_path='/kaggle/input/adult-census-income/adult.csv'):
    df = pd.read_csv(csv_path)
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df['income'] = np.where(df['income']=='>50K', 1, 0)
    # Upsample positives
    more_50k = df[df['income']==1]
    dist = more_50k.groupby(['race','sex']).size().reset_index(name='count')
    dist['up_count'] = (dist['count']*22654/dist['count'].sum()).round().astype(int)
    ups = []
    for _, row in dist.iterrows():
        group = more_50k[(more_50k['race']==row['race']) & (more_50k['sex']==row['sex'])]
        ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    df = pd.concat([df[df['income']==0]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Sensitive labels
    race_map = {"White": 0,"Black": 1,"Asian-Pac-Islander": 2,"Amer-Indian-Eskimo": 3,"Other": 4}
    sex_map = {"Male": 0,"Female": 1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    # Columns
    categorical_cols = ['workclass','education','marital.status','occupation','relationship','native.country']
    numeric_cols = ['age','fnlwgt','education.num','capital.gain','capital.loss','hours.per.week']
    # Preprocessor for NN
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    # BBN processing
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    # Split train/test
    X = df.drop(columns=['income','race_label','sex_label'])
    y = df['income'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    # BBN datasets aligned
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }

In [12]:
def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/compas/compas-scores-two-years.csv', seed=42):
    # Load and filter COMPAS data
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex',
         'priors_count','days_b_screening_arrest','decile_score',
         'juv_other_count','juv_misd_count','juv_fel_count',
         'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    # Sensitive label mapping
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    # Jail time calculation
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    # Target variable
    y = df['two_year_recid'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    # Columns
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    # Preprocessor for NN (scaled + one-hot)
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    # BBN preprocessing: discretize numerics, encode categoricals
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    # Split train/test
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    # BBN aligned splits
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }

In [13]:
def preprocess_german_for_fair_bbn(csv_path="/kaggle/input/credit-risk-analysis-dataset/german.data", seed=42):
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    df = pd.read_csv(csv_path, sep=' ', names=column_names)
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label'] = df['sex'].map({'male':0,'female':1})
    df['age_label'] = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label'] = df['credit'].map({1:1,2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    def upsample_minority_groups_german(data, target_col, group_cols, target_value, total_target_samples):
        minority = data[data[target_col]==target_value]
        dist = minority.groupby(group_cols).size().reset_index(name='count')
        dist['up_count'] = (dist['count']*total_target_samples/dist['count'].sum()).round().astype(int)
        ups = []
        for _, row in dist.iterrows():
            group = minority
            for col in group_cols:
                group = group[group[col]==row[col]]
            ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
        return pd.concat([data[data[target_col]!=target_value]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    df = upsample_minority_groups_german(
        data=df,
        target_col='credit_label',
        group_cols=['sex_label','age_label','foreign_worker_label'],
        target_value=0,
        total_target_samples=700
    )
    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    sensitive_sex = df['sex_label'].values
    sensitive_age = df['age_label'].values
    sensitive_foreign = df['foreign_worker_label'].values
    numerical_cols = ["duration", "amount", "installment_rate", "residence","number_credits","people_liable"]
    categorical_cols = [col for col in X.columns if col not in numerical_cols]
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    X_train_raw, X_test_raw, y_train, y_test, sens_age_train, sens_age_test, sens_sex_train, sens_sex_test, sens_foreign_train, sens_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=seed, stratify=y
    )
    pipeline = Pipeline([('preprocessor', preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)
    bbn_train_df = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test_df = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train_df, 'bbn_test_df': bbn_test_df,
        'y_train': y_train, 'y_test': y_test,
        'sens_age_train': sens_age_train, 'sens_age_test': sens_age_test,
        'sens_sex_train': sens_sex_train, 'sens_sex_test': sens_sex_test,
        'sens_foreign_train': sens_foreign_train, 'sens_foreign_test': sens_foreign_test
    }

In [14]:
def preprocess_lawschool_for_fair_bbn(law_path="/kaggle/input/law-school-admissions-bar-passage/bar_pass_prediction.csv"):
    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    target_col, sens_race, sens_gender = 'pass_bar', 'race', 'sex'
    # Drop rows missing critical columns
    law_df = law_df.dropna(subset=[target_col, sens_race, sens_gender])
    # Fill numeric NaNs with median
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = law_df[col].fillna(law_df[col].median())
    # Clip numeric outliers
    for col in law_df.select_dtypes(include=[np.number]).columns:
        q1, q3 = law_df[col].quantile(0.25), law_df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr > 0:
            law_df[col] = np.clip(law_df[col], q1 - 1.5*iqr, q3 + 1.5*iqr)
    # Binary income label
    law_df['income'] = np.where(law_df['fam_inc'] > law_df['fam_inc'].median(), 1, 0)
    # Upsample high income group
    high_income = law_df[law_df['income'] == 1]
    dist = high_income.groupby([sens_race, sens_gender]).size().reset_index(name='count')
    target_count = len(law_df[law_df['income'] == 0])
    dist['up_count'] = (dist['count'] * target_count / dist['count'].sum()).round().astype(int)
    ups = []
    for _, row in dist.iterrows():
        group = high_income[(high_income[sens_race] == row[sens_race]) & (high_income[sens_gender] == row[sens_gender])]
        if len(group) > 0:
            ups.append(resample(group, replace=True, n_samples=int(row['up_count']), random_state=seed))
    if ups:
        law_df = pd.concat([law_df[law_df['income'] == 0]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Encode sensitive attributes
    law_df[sens_race] = law_df[sens_race].astype('category')
    law_df[sens_gender] = law_df[sens_gender].astype('category')
    law_df['race_label'] = law_df[sens_race].cat.codes
    law_df['gender_label'] = law_df[sens_gender].cat.codes
    # Column groups
    numeric_cols = [c for c in ['lsat','ugpa','zfygpa','zgpa','age','gpa','fam_inc'] if c in law_df.columns]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime'] if c in law_df.columns]
    # Drop low variance cols
    low_var_cols = [col for col in numeric_cols if law_df[col].nunique() <= 1]
    if low_var_cols:
        print(f"Dropping low-variance columns: {low_var_cols}")
        law_df = law_df.drop(columns=low_var_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var_cols]
    # Preprocessor
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    # Discretized copy for BBN
    bbn_df = law_df.copy()
    for col in numeric_cols:
        if law_df[col].nunique() > 1:
            try:
                bbn_df[col] = pd.qcut(law_df[col], 5, labels=False, duplicates='drop')
            except:
                bbn_df[col] = pd.cut(law_df[col], 5, labels=False, duplicates='drop')
        else:
            bbn_df[col] = 0
        bbn_df[col] = bbn_df[col].fillna(0).astype(int)
    for col in categorical_cols + [sens_race, sens_gender]:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    X = law_df[numeric_cols + categorical_cols + [sens_race, sens_gender]]
    y = law_df['income'].values
    sens_race_labels = law_df['race_label']
    sens_gender_labels = law_df['gender_label']
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race_labels, sens_gender_labels, test_size=0.2, stratify=y, random_state=seed)
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    print(f"Final dataset — X_train_nn: {X_train_nn.shape}, BBN_train: {bbn_train.shape}")
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_gender_train': sens_gender_train.reset_index(drop=True),
        'sens_gender_test': sens_gender_test.reset_index(drop=True)
    }

In [15]:
# =============================================================
# 🧠 DIAGNOSTIC SCRIPT — UCI Bank Marketing Dataset
# Path: /kaggle/input/uci-ml-repository-bank-marketing/bank/bank-full.csv
# =============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from scipy import stats

# Load dataset safely
path = "/kaggle/input/uci-ml-repository-bank-marketing/bank/bank-full.csv"
try:
    df = pd.read_csv(path, sep=';')
except:
    df = pd.read_csv(path, sep=',')

print("============================================================")
print("📘 Dataset Loaded")
print("============================================================")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# -------------------------------------------------------------
# 1️⃣ Basic structure
# -------------------------------------------------------------
print("\n--- Basic Info ---")
print(df.info())

# -------------------------------------------------------------
# 2️⃣ Missing or 'unknown' values
# -------------------------------------------------------------
print("\n--- Missing Values ---")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\n--- 'unknown' Value Counts ---")
unknown_counts = (df == 'unknown').sum()
print(unknown_counts[unknown_counts > 0])

# -------------------------------------------------------------
# 3️⃣ Target column analysis
# -------------------------------------------------------------
target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
if target_col not in df.columns:
    target_col = df.columns[-1]
print(f"\n✅ Using '{target_col}' as target column")

if df[target_col].dtype == 'object':
    df['y_bin'] = np.where(df[target_col].str.lower().isin(['yes','y','true','1']), 1, 0)
else:
    df['y_bin'] = df[target_col]

print(df['y_bin'].value_counts(normalize=True).rename('ratio'))
print(f"Class imbalance ratio: {df['y_bin'].mean():.3f}")

# -------------------------------------------------------------
# 4️⃣ Sensitive attributes distribution (job, marital)
# -------------------------------------------------------------
for col in ['job', 'marital']:
    if col in df.columns:
        print(f"\n--- {col.upper()} distribution ---")
        print(df[col].value_counts(normalize=True).round(3))
    else:
        print(f"⚠️ Missing sensitive column: {col}")

# -------------------------------------------------------------
# 5️⃣ Outlier check for numeric columns
# -------------------------------------------------------------
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    num_cols = ['age','balance','day','duration','campaign','pdays','previous']
    num_cols = [c for c in num_cols if c in df.columns]

print(f"\nNumeric columns: {num_cols}")

iqr_outliers = {}
for col in num_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = 100 * outliers / len(df)
    iqr_outliers[col] = pct
print("\n--- Outlier percentage per numeric feature ---")
for col, pct in iqr_outliers.items():
    print(f"{col}: {pct:.2f}%")

# -------------------------------------------------------------
# 6️⃣ Correlation / skew check
# -------------------------------------------------------------
print("\n--- Correlation Matrix (top features) ---")
try:
    corr = df[num_cols + ['y_bin']].corr(numeric_only=True)
    if isinstance(corr, pd.DataFrame):
        y_corr = corr.loc[:, 'y_bin'].sort_values(ascending=False)
        print(y_corr)
    else:
        print("⚠️ Correlation result is not a DataFrame, skipping detailed view.")
except Exception as e:
    print(f"⚠️ Correlation computation failed: {e}")

# -------------------------------------------------------------
# 7️⃣ Categorical imbalance (jobs, marital × y)
# -------------------------------------------------------------
print("\n--- Target vs Job and Marital ---")
if 'job' in df.columns and 'marital' in df.columns:
    pivot = df.pivot_table(index='job', columns='marital', values='y_bin', aggfunc='mean')
    print(pivot.fillna(0).round(2))
else:
    print("⚠️ One or both columns missing for fairness grouping.")

# -------------------------------------------------------------
# 8️⃣ Recommendations summary
# -------------------------------------------------------------
print("\n============================================================")
print("🧭 PREPROCESSING RECOMMENDATION SUMMARY")
print("============================================================")

if unknown_counts.sum() > 0:
    print("✅ Replace or drop 'unknown' values (they appear in key categorical columns).")
else:
    print("❌ No 'unknown' values — label cleanup not required.")

if df['y_bin'].mean() < 0.3 or df['y_bin'].mean() > 0.7:
    print("✅ Upsampling or downsampling needed (class imbalance detected).")
else:
    print("❌ Target roughly balanced — resampling not critical.")

if any(pct > 5 for pct in iqr_outliers.values()):
    print("✅ Consider clipping outliers (balance/duration/pdays show heavy tails).")
else:
    print("❌ Outliers minimal — clipping optional.")

if 'job' in df.columns and 'marital' in df.columns:
    print("✅ Sensitive attributes 'job' and 'marital' suitable for fairness splits.")
else:
    print("⚠️ Missing sensitive columns — fairness mapping may fail.")

print("✅ Convert all categorical columns to lowercase strings before encoding.")
print("✅ Ensure 'y' is binary (yes/no → 1/0).")
print("✅ Standardize numeric columns before neural model.")
print("✅ Discretize numeric columns (qcut) for BBN use.")

print("\n✅ Diagnostic complete.")


📘 Dataset Loaded
Shape: (45211, 17)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

--- Basic Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  i

In [16]:
def preprocess_bank_for_fair_bbn(csv_path='/kaggle/input/uci-ml-repository-bank-marketing/bank/bank-full.csv', seed=42):
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.model_selection import train_test_split
    from sklearn.utils import resample
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
    if target_col not in df.columns:
        target_col = df.columns[-1]
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes', 'y', 'true', '1']), 1, 0)
    positives = df[df['y'] == 1]
    negatives = df[df['y'] == 0]
    if len(positives) == 0:
        df.loc[df.head(100).index, 'y'] = 1
        positives = df[df['y'] == 1]
        negatives = df[df['y'] == 0]
    dist = positives.groupby(['job', 'marital']).size().reset_index(name='count')
    dist['up_count'] = (dist['count'] * len(negatives) / dist['count'].sum()).round().astype(int)
    upsamples = []
    for _, row in dist.iterrows():
        group = positives[(positives['job'] == row['job']) & (positives['marital'] == row['marital'])]
        if len(group) > 0:
            upsamples.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    df = pd.concat([negatives] + upsamples).sample(frac=1, random_state=seed).reset_index(drop=True)
    le_marital = LabelEncoder()
    df['marital_label'] = le_marital.fit_transform(df['marital'])
    le_job = LabelEncoder()
    df['job_label'] = le_job.fit_transform(df['job'])
    categorical_cols = [col for col in ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] if col in df.columns]
    numeric_cols = [col for col in ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] if col in df.columns]
    for col in ['balance', 'duration', 'pdays', 'previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    X = df.drop(columns=['y', 'marital_label', 'job_label'])
    y = df['y'].values
    sens_marital = df['marital_label']
    sens_job = df['job_label']
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed)
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }

In [17]:
class SimpleFairBBN:
    def __init__(self, feature_names=None, target_name='y'):
        self.feature_names = feature_names or []
        self.target_name = target_name
        self.model = None
        self.inference = None

    def fit(self, df_discrete, y, include_sensitive=True):
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y
        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:6]
        if include_sensitive:
            for sens in ['marital', 'job', 'race', 'sex']:
                if sens in df_discrete.columns and sens not in selected:
                    selected.append(sens)
        edges = [(f, self.target_name) for f in selected]
        self.feature_names = selected
        self.model = DiscreteBayesianNetwork(edges)
        self.model.fit(data, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        Xdf = df_discrete.reset_index(drop=True)
        probs = []
        for _, row in Xdf.iterrows():
            evidence = {}
            used = 0
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used < 3:
                    try:
                        evidence[f] = int(row[f])
                        used += 1
                    except:
                        continue
            try:
                q = self.inference.query(variables=[self.target_name], evidence=evidence) if evidence else self.inference.query(variables=[self.target_name])
                p1 = q.values[1] if len(q.values) == 2 else 0.5
            except:
                p1 = 0.5
            probs.append(p1)
        return np.array(probs)

In [18]:
class AdapterNN(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x, return_features=False):
        h = self.act(self.fc1(x))
        h2 = self.act(self.fc2(h))
        logit = self.out(h2)
        return (logit, h2) if return_features else logit

In [19]:
class AdversaryNN(nn.Module):
    def __init__(self, in_dim, marital_classes, job_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.marital_head = nn.Linear(32, marital_classes)
        self.job_head = nn.Linear(32, job_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.marital_head(s), self.job_head(s)

class AdversaryNN_MEPS(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_LawSchool(nn.Module):
    def __init__(self, in_dim, race_classes, gender_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.gender_head = nn.Linear(32, gender_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.gender_head(s)

class AdversaryNN_Adult(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_COMPAS(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_German(nn.Module):
    def __init__(self, in_dim, age_classes, sex_classes, foreign_classes):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU()
        )
        self.age_head = nn.Linear(32, age_classes)
        self.sex_head = nn.Linear(32, sex_classes)
        self.foreign_head = nn.Linear(32, foreign_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.age_head(s), self.sex_head(s), self.foreign_head(s)

In [20]:
def train_fair_bbn_adapter_german(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    X_train_nn = data_dict['X_train_nn']; X_test_nn = data_dict['X_test_nn']
    bbn_train_df = data_dict['bbn_train_df']; bbn_test_df = data_dict['bbn_test_df']
    y_train = data_dict['y_train']; y_test = data_dict['y_test']
    sens_age_train = data_dict['sens_age_train']; sens_age_test = data_dict['sens_age_test']
    sens_sex_train = data_dict['sens_sex_train']; sens_sex_test = data_dict['sens_sex_test']
    sens_foreign_train = data_dict['sens_foreign_train']; sens_foreign_test = data_dict['sens_foreign_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_age_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_age_test)
    base_age_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_age_test)
    base_foreign_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_foreign_test)
    base_foreign_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_foreign_test)

    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training Fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns))
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_age = bbn.predict_proba(bbn_train_df[['age_label']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex_label']]).reshape(-1,1)
    p_foreign = bbn.predict_proba(bbn_train_df[['foreign_worker_label']]).reshape(-1,1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_age, p_sex, p_foreign]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_age_test = bbn.predict_proba(bbn_test_df[['age_label']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex_label']]).reshape(-1,1)
    p_foreign_test = bbn.predict_proba(bbn_test_df[['foreign_worker_label']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_age_test, p_sex_test, p_foreign_test]), dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    age_train_t = torch.tensor(sens_age_train.astype(int), dtype=torch.long)
    sex_train_t = torch.tensor(sens_sex_train.astype(int), dtype=torch.long)
    foreign_train_t = torch.tensor(sens_foreign_train.astype(int), dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(adapter_in_train, y_train_t, age_train_t, sex_train_t, foreign_train_t),
        batch_size=batch_size, shuffle=True
    )

    adapter = AdapterNN(in_dim=4, hidden_dim=64)
    adversary = AdversaryNN_German(in_dim=32, age_classes=2, sex_classes=2, foreign_classes=2)

    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training Adapter with Adversarial + Fairness Regularization...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss = 0.0; total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y, batch_age, batch_sex, batch_foreign = batch

            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            age_logits, sex_logits, foreign_logits = adversary(features)
            adv_loss = (
                adv_loss_fn(age_logits, batch_age) +
                adv_loss_fn(sex_logits, batch_sex) +
                adv_loss_fn(foreign_logits, batch_foreign)
            ) / 3.0
            adv_loss.backward(); adv_opt.step()
            total_adv_loss += adv_loss.item()

            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            age_logits2, sex_logits2, foreign_logits2 = adversary(features)
            adv_loss_for_adapter = (
                adv_loss_fn(age_logits2, batch_age) +
                adv_loss_fn(sex_logits2, batch_sex) +
                adv_loss_fn(foreign_logits2, batch_foreign)
            ) / 3.0

            dp_penalty = torch.abs(features[batch_sex==0].mean() - features[batch_sex==1].mean())
            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_age_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_age_test)
    adv_age_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_age_test)
    adv_foreign_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_foreign_test)
    adv_foreign_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_foreign_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Sex DP: {base_sex_dp:.4f}, Sex EO: {base_sex_eo:.4f}")
    print(f" Age DP: {base_age_dp:.4f}, Age EO: {base_age_eo:.4f}")
    print(f" Foreign DP: {base_foreign_dp:.4f}, Foreign EO: {base_foreign_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Sex DP: {adv_sex_dp:.4f}, Sex EO: {adv_sex_eo:.4f}")
    print(f" Age DP: {adv_age_dp:.4f}, Age EO: {adv_age_eo:.4f}")
    print(f" Foreign DP: {adv_foreign_dp:.4f}, Foreign EO: {adv_foreign_eo:.4f}")

    return {
        'baseline': {
            'pred': base_pred, 'acc': base_acc,
            'sex_dp': base_sex_dp, 'sex_eo': base_sex_eo,
            'age_dp': base_age_dp, 'age_eo': base_age_eo,
            'foreign_dp': base_foreign_dp, 'foreign_eo': base_foreign_eo
        },
        'bbn_adapter': {
            'pred': test_pred, 'acc': adv_acc,
            'sex_dp': adv_sex_dp, 'sex_eo': adv_sex_eo,
            'age_dp': adv_age_dp, 'age_eo': adv_age_eo,
            'foreign_dp': adv_foreign_dp, 'foreign_eo': adv_foreign_eo
        }
    }
    


In [21]:

def train_fair_bbn_adapter_compas(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    """Training function for COMPAS dataset"""
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_race_train, sens_race_test = data_dict['sens_race_train'], data_dict['sens_race_test']
    sens_sex_train, sens_sex_test = data_dict['sens_sex_train'], data_dict['sens_sex_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns), target_name='two_year_recid')
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)
    
    # Generate marginals for adapter input
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_race = bbn.predict_proba(bbn_train_df[['race']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex']]).reshape(-1,1)
    adapter_in_train = torch.tensor(np.hstack([p_all,p_race,p_sex]), dtype=torch.float32)
    
    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_race_test = bbn.predict_proba(bbn_test_df[['race']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test,p_race_test,p_sex_test]), dtype=torch.float32)

    # Convert labels & sensitive attrs
    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    sex_train_t  = torch.tensor(sens_sex_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(adapter_in_train, y_train_t, race_train_t, sex_train_t),
        batch_size=batch_size, shuffle=True
    )

    # Initialize adapter + adversary
    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN_COMPAS(in_dim=32,
                            race_classes=len(sens_race_train.unique()),
                            sex_classes=len(sens_sex_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    # Training loop
    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss=0.0; total_adv_loss=0.0
        for batch_in, batch_y, batch_race, batch_sex in train_loader:
            # --- Train adversary ---
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, s_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits,batch_race) + adv_loss_fn(s_logits,batch_sex))/2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()
    
            # --- Train adapter ---
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit,batch_y)
            r_logits2, s_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2,batch_race) + adv_loss_fn(s_logits2,batch_sex))/2.0
            # fairness penalty across race groups
            dp_penalty = torch.abs(features[batch_race==0].mean() - features[batch_race==1].mean())
            total_loss = pred_loss - lambda_adv*adv_loss_for_adapter + alpha_bbn*dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()
    
        if epoch % 10==0 or epoch==epochs-1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")
    
    # Evaluation
    adapter.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs>0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)

    # Print results
    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Sex  DP: {base_sex_dp:.4f}, Sex  EO: {base_sex_eo:.4f}")
    
    print("\nBBN + Adapter (Adversarial + Fairness) RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Sex  DP: {adv_sex_dp:.4f}, Sex  EO: {adv_sex_eo:.4f}")
    
    return {
        'baseline':{
            'pred':base_pred,'acc':base_acc,
            'race_dp':base_race_dp,'race_eo':base_race_eo,
            'sex_dp':base_sex_dp,'sex_eo':base_sex_eo
        },
        'bbn_adapter':{
            'pred':test_pred,'acc':adv_acc,
            'race_dp':adv_race_dp,'race_eo':adv_race_eo,
            'sex_dp':adv_sex_dp,'sex_eo':adv_sex_eo
        }
    }
    


In [22]:

def train_fair_bbn_adapter_adult(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    """Training function for Adult dataset"""
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_race_train, sens_race_test = data_dict['sens_race_train'], data_dict['sens_race_test']
    sens_sex_train, sens_sex_test = data_dict['sens_sex_train'], data_dict['sens_sex_test']
    
    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns))
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)

    # Generate BBN marginals
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_race = bbn.predict_proba(bbn_train_df[['race']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex']]).reshape(-1,1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_race, p_sex]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_race_test = bbn.predict_proba(bbn_test_df[['race']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_race_test, p_sex_test]), dtype=torch.float32)

    # Convert to tensors
    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    sex_train_t = torch.tensor(sens_sex_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, race_train_t, sex_train_t), 
                              batch_size=batch_size, shuffle=True)

    # Models - USING ADULT-SPECIFIC ADVERSARY
    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN_Adult(in_dim=32, race_classes=len(sens_race_train.unique()), sex_classes=len(sens_sex_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss=0.0; total_adv_loss=0.0
        for batch in train_loader:
            batch_in, batch_y, batch_race, batch_sex = batch

            # Train adversary
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, s_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits,batch_race) + adv_loss_fn(s_logits,batch_sex))/2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss+=adv_loss.item()

            # Train adapter
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit,batch_y)
            r_logits2, s_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2,batch_race) + adv_loss_fn(s_logits2,batch_sex))/2.0

            # BBN fairness regularization
            dp_penalty = torch.abs(features[batch_race==0].mean() - features[batch_race==1].mean())
            total_loss = pred_loss - lambda_adv*adv_loss_for_adapter + alpha_bbn*dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10==0 or epoch==epochs-1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    # Evaluation
    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs>0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Sex  DP: {base_sex_dp:.4f}, Sex  EO: {base_sex_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Sex  DP: {adv_sex_dp:.4f}, Sex  EO: {adv_sex_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc, 'race_dp': base_race_dp, 'race_eo': base_race_eo,
                     'sex_dp': base_sex_dp, 'sex_eo': base_sex_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc, 'race_dp': adv_race_dp, 'race_eo': adv_race_eo,
                        'sex_dp': adv_sex_dp, 'sex_eo': adv_sex_eo}
    }


In [24]:
def train_fair_bbn_adapter_bank(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_marital_train, sens_marital_test = data_dict['sens_marital_train'], data_dict['sens_marital_test']
    sens_job_train, sens_job_test = data_dict['sens_job_train'], data_dict['sens_job_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_marital_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_marital_test)
    base_marital_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_marital_test)
    base_job_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_job_test)
    base_job_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_job_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    features_to_use = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'marital', 'job']
    bbn_train_sub = bbn_train_df[features_to_use]
    bbn_test_sub = bbn_test_df[features_to_use]

    bbn = SimpleFairBBN(feature_names=features_to_use, target_name='y')
    bbn.fit(bbn_train_sub, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_sub).reshape(-1, 1)
    p_marital = bbn.predict_proba(bbn_train_sub[['marital']]).reshape(-1, 1)
    p_job = bbn.predict_proba(bbn_train_sub[['job']]).reshape(-1, 1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_marital, p_job]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_sub).reshape(-1, 1)
    p_marital_test = bbn.predict_proba(bbn_test_sub[['marital']]).reshape(-1, 1)
    p_job_test = bbn.predict_proba(bbn_test_sub[['job']]).reshape(-1, 1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_marital_test, p_job_test]), dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)
    marital_train_t = torch.tensor(sens_marital_train.values.astype(int), dtype=torch.long)
    job_train_t = torch.tensor(sens_job_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, marital_train_t, job_train_t),
                              batch_size=batch_size, shuffle=True)

    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN(in_dim=32, marital_classes=len(sens_marital_train.unique()), job_classes=len(sens_job_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss = 0.0; total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y, batch_marital, batch_job = batch

            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            m_logits, j_logits = adversary(features)
            adv_loss = (adv_loss_fn(m_logits, batch_marital) + adv_loss_fn(j_logits, batch_job)) / 2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()

            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            m_logits2, j_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(m_logits2, batch_marital) + adv_loss_fn(j_logits2, batch_job)) / 2.0

            grp0_mean = features[batch_marital == 0].mean(dim=0)
            grp1_mean = features[batch_marital == 1].mean(dim=0)
            dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss / len(train_loader):.4f} | Adapter Loss: {total_adapter_loss / len(train_loader):.4f}")

    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_marital_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_marital_test)
    adv_marital_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_marital_test)
    adv_job_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_job_test)
    adv_job_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_job_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Marital DP: {base_marital_dp:.4f}, Marital EO: {base_marital_eo:.4f}")
    print(f" Job     DP: {base_job_dp:.4f}, Job     EO: {base_job_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Marital DP: {adv_marital_dp:.4f}, Marital EO: {adv_marital_eo:.4f}")
    print(f" Job     DP: {adv_job_dp:.4f}, Job     EO: {adv_job_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc, 'marital_dp': base_marital_dp, 'marital_eo': base_marital_eo,
                     'job_dp': base_job_dp, 'job_eo': base_job_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc, 'marital_dp': adv_marital_dp, 'marital_eo': adv_marital_eo,
                        'job_dp': adv_job_dp, 'job_eo': adv_job_eo}
    }

In [25]:
def train_fair_bbn_adapter_lawschool(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_race_train, sens_race_test = data_dict['sens_race_train'], data_dict['sens_race_test']
    sens_gender_train, sens_gender_test = data_dict['sens_gender_train'], data_dict['sens_gender_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_gender_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_gender_test)
    base_gender_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_gender_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    features_to_use = ['lsat', 'ugpa', 'fulltime', 'fam_inc', 'age', 'race', 'gender']
    features_to_use = [f for f in features_to_use if f in bbn_train_df.columns]
    bbn_train_sub = bbn_train_df[features_to_use]
    bbn_test_sub = bbn_test_df[features_to_use]

    bbn = SimpleFairBBN(feature_names=features_to_use, target_name='pass_bar')
    bbn.fit(bbn_train_sub, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_sub).reshape(-1, 1)
    p_race = bbn.predict_proba(bbn_train_sub[['race']]).reshape(-1, 1)
    p_gender = bbn.predict_proba(bbn_train_sub[['gender']]).reshape(-1, 1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_race, p_gender]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_sub).reshape(-1, 1)
    p_race_test = bbn.predict_proba(bbn_test_sub[['race']]).reshape(-1, 1)
    p_gender_test = bbn.predict_proba(bbn_test_sub[['gender']]).reshape(-1, 1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_race_test, p_gender_test]), dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    gender_train_t = torch.tensor(sens_gender_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, race_train_t, gender_train_t),
                              batch_size=batch_size, shuffle=True)

    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN_LawSchool(in_dim=32,
                                      race_classes=len(sens_race_train.unique()),
                                      gender_classes=len(sens_gender_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss, total_adv_loss = 0.0, 0.0
        for batch in train_loader:
            batch_in, batch_y, batch_race, batch_gender = batch

            # Train adversary
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, g_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits, batch_race) + adv_loss_fn(g_logits, batch_gender)) / 2
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()

            # Train adapter
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            r_logits2, g_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2, batch_race) + adv_loss_fn(g_logits2, batch_gender)) / 2

            grp0_mean = features[batch_race == 0].mean(dim=0)
            grp1_mean = features[batch_race == 1].mean(dim=0)
            dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward(); adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:03d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    adapter.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_pred = (torch.sigmoid(test_logit).cpu().numpy().flatten() > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_gender_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_gender_test)
    adv_gender_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_gender_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Gender DP: {base_gender_dp:.4f}, Gender EO: {base_gender_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Gender DP: {adv_gender_dp:.4f}, Gender EO: {adv_gender_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc,
                     'race_dp': base_race_dp, 'race_eo': base_race_eo,
                     'gender_dp': base_gender_dp, 'gender_eo': base_gender_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc,
                        'race_dp': adv_race_dp, 'race_eo': adv_race_eo,
                        'gender_dp': adv_gender_dp, 'gender_eo': adv_gender_eo}
    }



In [26]:
if __name__ == '__main__':
    print("🚀 Starting Fair BBN System on Kaggle")
    print("=" * 60)

    datasets = [
        ("GERMAN CREDIT", preprocess_german_for_fair_bbn, train_fair_bbn_adapter_german),
        ("COMPAS", preprocess_compas_for_fair_bbn, train_fair_bbn_adapter_compas),
        ("ADULT INCOME", preprocess_adult_for_fair_bbn, train_fair_bbn_adapter_adult),
        ("BANK", preprocess_bank_for_fair_bbn, train_fair_bbn_adapter_bank),
        ("LAWSCHOOL", preprocess_lawschool_for_fair_bbn, train_fair_bbn_adapter_lawschool),
    ]

    for name, preprocess_func, train_func in datasets:
        print(f"\n{'=' * 60}\n▶ PROCESSING {name} DATASET\n{'=' * 60}")
        data = preprocess_func()
        results = train_func(data)
        print(f"✅ {name} dataset completed successfully")

    print("\n" + "=" * 60)
    print("🎯 FAIR BBN SYSTEM EXECUTION COMPLETE")
    print("=" * 60)


🚀 Starting Fair BBN System on Kaggle

▶ PROCESSING GERMAN CREDIT DATASET
Training baseline MLP...
Baseline MLP Accuracy: 0.8571
Training Fair BBN...
Training Adapter with Adversarial + Fairness Regularization...
Epoch   0 | Adv Loss: 0.6125 | Adapter Loss: 0.3911
Epoch  10 | Adv Loss: 0.4823 | Adapter Loss: 0.4565
Epoch  20 | Adv Loss: 0.3874 | Adapter Loss: 0.5010
Epoch  30 | Adv Loss: 0.3206 | Adapter Loss: 0.5333
Epoch  40 | Adv Loss: 0.3042 | Adapter Loss: 0.5415
Epoch  50 | Adv Loss: 0.2974 | Adapter Loss: 0.5446
Epoch  59 | Adv Loss: 0.2842 | Adapter Loss: 0.5510

BASELINE MLP RESULTS:
 Accuracy: 0.8571
 Sex DP: 0.0461, Sex EO: 0.0945
 Age DP: 0.0940, Age EO: 0.0731
 Foreign DP: 0.4361, Foreign EO: 0.2188

BBN + Adapter RESULTS:
 Accuracy: 0.5000
 Sex DP: 0.0000, Sex EO: 0.0000
 Age DP: 0.0000, Age EO: 0.0000
 Foreign DP: 0.0000, Foreign EO: 0.0000
✅ GERMAN CREDIT dataset completed successfully

▶ PROCESSING COMPAS DATASET


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


Training baseline MLP...
Baseline MLP Accuracy: 0.6372
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.2671 | Adapter Loss: 0.0715
Epoch  10 | Adv Loss: 0.8447 | Adapter Loss: 0.2710
Epoch  20 | Adv Loss: 0.8083 | Adapter Loss: 0.2859
Epoch  30 | Adv Loss: 0.8049 | Adapter Loss: 0.2868
Epoch  40 | Adv Loss: 0.8009 | Adapter Loss: 0.2882
Epoch  50 | Adv Loss: 0.8063 | Adapter Loss: 0.2856
Epoch  59 | Adv Loss: 0.8036 | Adapter Loss: 0.2866

BASELINE MLP RESULTS:
 Accuracy: 0.6372
 Race DP: 0.5538, Race EO: 0.6558
 Sex  DP: 0.1319, Sex  EO: 0.1136

BBN + Adapter (Adversarial + Fairness) RESULTS:
 Accuracy: 0.5449
 Race DP: 0.0000, Race EO: 0.0000
 Sex  DP: 0.0000, Sex  EO: 0.0000
✅ COMPAS dataset completed successfully

▶ PROCESSING ADULT INCOME DATASET
Training baseline MLP...


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Baseline MLP Accuracy: 0.8746
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 0.9057 | Adapter Loss: 0.2431
Epoch  10 | Adv Loss: 0.5357 | Adapter Loss: 0.4253
Epoch  20 | Adv Loss: 0.5355 | Adapter Loss: 0.4254
Epoch  30 | Adv Loss: 0.5354 | Adapter Loss: 0.4255
Epoch  40 | Adv Loss: 0.5357 | Adapter Loss: 0.4253
Epoch  50 | Adv Loss: 0.5356 | Adapter Loss: 0.4254
Epoch  59 | Adv Loss: 0.5356 | Adapter Loss: 0.4254

BASELINE MLP RESULTS:
 Accuracy: 0.8746
 Race DP: 0.3269, Race EO: 0.1584
 Sex  DP: 0.3241, Sex  EO: 0.1649

BBN + Adapter RESULTS:
 Accuracy: 0.5000
 Race DP: 0.0000, Race EO: 0.0000
 Sex  DP: 0.0000, Sex  EO: 0.0000
✅ ADULT INCOME dataset completed successfully

▶ PROCESSING BANK DATASET
Training baseline MLP...


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Baseline MLP Accuracy: 0.9290
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.7198 | Adapter Loss: -0.1652
Epoch  10 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  20 | Adv Loss: 1.5220 | Adapter Loss: -0.0678
Epoch  30 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  40 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  50 | Adv Loss: 1.5222 | Adapter Loss: -0.0679
Epoch  59 | Adv Loss: 1.5221 | Adapter Loss: -0.0679

BASELINE MLP RESULTS:
 Accuracy: 0.9290
 Marital DP: 0.1137, Marital EO: 0.0642
 Job     DP: 0.4521, Job     EO: 0.3489

BBN + Adapter RESULTS:
 Accuracy: 0.5002
 Marital DP: 0.0000, Marital EO: 0.0000
 Job     DP: 0.0000, Job     EO: 0.0000
✅ BANK dataset completed successfully

▶ PROCESSING LAWSCHOOL DATASET
Final dataset — X_train_nn: (32916, 23), BBN_train: (32916, 42)
Training baseline MLP...
Baseline MLP Accuracy: 1.0000
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch 000 | Adv Loss: